# Network Planning Tower – Interactive Power BI Visualization

Loads the curated star schema CSVs from `data/curated/`, merges them into a
single flattened DataFrame, calculates `Net_Position`, then renders an
interactive Power BI report via `powerbiclient.QuickVisualize`.

**Run order:** Cell 1 → Cell 2 → Cell 3  
**Requirement:** Power BI Pro or Premium-per-user licence on the authenticating account.

In [1]:
# ── Cell 1 · Imports ─────────────────────────────────────────────────────────
import os
import pandas as pd

from powerbiclient import QuickVisualize, get_dataset_config
from powerbiclient.authentication import DeviceCodeLoginAuthentication

In [2]:
# ── Cell 2 · Load, merge, and enrich curated data ────────────────────────────
#
# Grain alignment note
# --------------------
# Fact_Inventory  : DateKey × SKUKey × LocationKey          (daily snapshot – base)
# Fact_Inbound    : DeliveryDateKey × SKUKey × LocationKey × PO line  (aggregate first)
# Fact_Demand     : ShipDateKey    × SKUKey × LocationKey × order line (aggregate first)
# Fact_Transfers  : empty in this snapshot (schema-ready, zero rows)
# All facts are joined to Dim_Product and Dim_Location via surrogate keys.

# ── 2.1  Resolve the curated/ folder relative to this notebook ───────────────
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
CURATED = os.path.normpath(os.path.join(NOTEBOOK_DIR, "..", "..", "data", "curated"))

def load(name):
    path = os.path.join(CURATED, f"{name}.csv")
    df = pd.read_csv(path)
    print(f"  {name}: {len(df):,} rows, {len(df.columns)} cols")
    return df

print("Loading curated CSVs ...")
fact_inv   = load("Fact_Inventory")   # planning position snapshot
fact_ib    = load("Fact_Inbound")     # PO-line receipts
fact_tr    = load("Fact_Transfers")   # outbound transfer movements
fact_dem   = load("Fact_Demand")      # sales order lines
dim_prod   = load("Dim_Product")
dim_loc    = load("Dim_Location")

# ── 2.2  Normalise date keys so all facts share the column name "DateKey" ─────
fact_ib  = fact_ib.rename(columns={"DeliveryDateKey": "DateKey"})
fact_dem = fact_dem.rename(columns={"ShipDateKey": "DateKey"})

# ── 2.3  Aggregate Fact_Inbound to DateKey × SKUKey × LocationKey ─────────────
# (raw grain is PO-line level; must match Fact_Inventory grain before joining)
ib_agg = (
    fact_ib
    .groupby(["DateKey", "SKUKey", "LocationKey"], as_index=False)
    .agg(
        Inbound_ExpectedQty  = ("ExpectedQty",  "sum"),
        Inbound_ReceivedQty  = ("ReceivedQty",  "sum"),
    )
)

# ── 2.4  Aggregate Fact_Demand to DateKey × SKUKey × LocationKey ──────────────
# (raw grain is sales-order-line level)
dem_agg = (
    fact_dem
    .groupby(["DateKey", "SKUKey", "LocationKey"], as_index=False)
    .agg(
        Demand_OriginalQty     = ("OriginalQty",     "sum"),
        Demand_OutstandingQty  = ("OutstandingQty",  "sum"),
    )
)

# ── 2.5  Aggregate Fact_Transfers (empty in this snapshot; future-proof) ──────
if not fact_tr.empty and "SKUCode" in fact_tr.columns:
    # Transfers use business codes, not surrogate keys – map to keys first
    code_to_key = dim_prod.set_index("SKUCode")["SKUKey"].to_dict()
    loc_to_key  = dim_loc.set_index("LocationCode")["LocationKey"].to_dict()
    fact_tr["SKUKey"]      = fact_tr["SKUCode"].map(code_to_key)
    fact_tr["LocationKey"] = fact_tr["OriginLocationCode"].map(loc_to_key)
    fact_tr["DateKey"]     = pd.to_datetime(fact_tr["ShipDate"]).dt.strftime("%Y%m%d").astype(int)
    tr_agg = (
        fact_tr
        .groupby(["DateKey", "SKUKey", "LocationKey"], as_index=False)
        .agg(Transfer_ShipQty = ("TransferQty", "sum"))
    )
else:
    tr_agg = pd.DataFrame(columns=["DateKey", "SKUKey", "LocationKey", "Transfer_ShipQty"])

# ── 2.6  Outer-join all facts on DateKey × SKUKey × LocationKey ───────────────
# Start from Fact_Inventory (the planning snapshot) as the spine.
# Outer joins preserve SKU/date combinations that appear in only one source.
join_keys = ["DateKey", "SKUKey", "LocationKey"]

merged = (
    fact_inv
    .merge(ib_agg,  on=join_keys, how="outer")
    .merge(dem_agg, on=join_keys, how="outer")
    .merge(tr_agg,  on=join_keys, how="outer")
)

# ── 2.7  Bring in descriptive attributes from dimensions ──────────────────────
merged = merged.merge(
    dim_prod[["SKUKey", "SKUCode", "ProductDescription", "Commodity", "ProductGroup"]],
    on="SKUKey", how="left"
)
merged = merged.merge(
    dim_loc[["LocationKey", "LocationCode", "LocationType"]],
    on="LocationKey", how="left"
)

# ── 2.8  Fill NaN in numeric columns with 0 ───────────────────────────────────
numeric_cols = merged.select_dtypes(include="number").columns
merged[numeric_cols] = merged[numeric_cols].fillna(0)

# ── 2.9  Calculate Net_Position ───────────────────────────────────────────────
#
# Formula requested:
#   Net_Position = (Starting_Inventory_Qty + Received_Qty + Transfer_In_Qty)
#                - (Good_Sales_Qty + Transfer_Out_Qty + Build_Rework_Qty + Donate_Piker_Qty)
#
# Column mapping to actual CSV names:
#   Starting_Inventory_Qty  → InventoryOnHand        (Fact_Inventory)
#   Received_Qty            → Inbound_ReceivedQty    (Fact_Inbound aggregated)
#   Transfer_In_Qty         → TransfersInReceived     (Fact_Inventory)
#   Good_Sales_Qty          → GoodSalesShipping       (Fact_Inventory)
#   Transfer_Out_Qty        → TransferShippingOut     (Fact_Inventory)
#   Build_Rework_Qty        → 0  (Production Out rows were not extracted as a
#                                 standalone column in this ETL run; they are
#                                 included inside TotalSupply from the planning
#                                 report but not isolated. Extend the ETL to
#                                 add METRIC["ProductionOut"] if needed.)
#   Donate_Piker_Qty        → Donations              (Fact_Inventory)
#
merged["Net_Position"] = (
    (merged["InventoryOnHand"]   + merged["Inbound_ReceivedQty"] + merged["TransfersInReceived"])
    - (merged["GoodSalesShipping"] + merged["TransferShippingOut"] + merged["Donations"])
)

# ── 2.10  Final tidy-up ───────────────────────────────────────────────────────
# Convert DateKey (int YYYYMMDD) to an actual date string for Power BI
merged["Date"] = pd.to_datetime(merged["DateKey"].astype(int).astype(str), format="%Y%m%d")

# Drop surrogate keys — Power BI QuickVisualize doesn't need raw integer FKs;
# keeping descriptive columns makes the auto-report more readable.
display_cols = [
    "Date",
    "SKUCode",
    "ProductDescription",
    "Commodity",
    "ProductGroup",
    "LocationCode",
    "LocationType",
    # Supply components
    "InventoryOnHand",
    "Inbound_ExpectedQty",
    "Inbound_ReceivedQty",
    "TransfersInExpected",
    "TransfersInReceived",
    "TotalSupply",
    # Demand components
    "GoodSalesShipping",
    "Demand_OriginalQty",
    "Demand_OutstandingQty",
    "Donations",
    "TransferShippingOut",
    # Derived
    "Net_Position",
    "NetAvailable",   # pre-computed by planning report (cross-check)
]
df_viz = merged[[c for c in display_cols if c in merged.columns]].copy()

print(f"\nFinal flattened DataFrame: {len(df_viz):,} rows × {len(df_viz.columns)} columns")
print("Columns:", list(df_viz.columns))
print("\nSample (5 rows):")
df_viz.head()

Loading curated CSVs ...
  Fact_Inventory: 1,185 rows, 13 cols
  Fact_Inbound: 1,366 rows, 9 cols
  Fact_Transfers: 0 rows, 5 cols
  Fact_Demand: 2,151 rows, 11 cols
  Dim_Product: 119 rows, 9 cols
  Dim_Location: 236 rows, 4 cols

Final flattened DataFrame: 2,791 rows × 20 columns
Columns: ['Date', 'SKUCode', 'ProductDescription', 'Commodity', 'ProductGroup', 'LocationCode', 'LocationType', 'InventoryOnHand', 'Inbound_ExpectedQty', 'Inbound_ReceivedQty', 'TransfersInExpected', 'TransfersInReceived', 'TotalSupply', 'GoodSalesShipping', 'Demand_OriginalQty', 'Demand_OutstandingQty', 'Donations', 'TransferShippingOut', 'Net_Position', 'NetAvailable']

Sample (5 rows):


,Date,SKUCode,ProductDescription,Commodity,ProductGroup,LocationCode,LocationType,InventoryOnHand,Inbound_ExpectedQty,Inbound_ReceivedQty,TransfersInExpected,TransfersInReceived,TotalSupply,GoodSalesShipping,Demand_OriginalQty,Demand_OutstandingQty,Donations,TransferShippingOut,Net_Position,NetAvailable
0,2026-03-16,RTBB1007,Beefsteak - 15lb 22ct Beef No1,Beefsteak,BB,DIRECT,DC,0.0,180.0,180.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,180.0,0.0
1,2026-03-16,RTBB1007,Beefsteak - 15lb 22ct Beef No1,Beefsteak,BB,MPL1,MPL,0.0,90.0,90.0,0.0,0.0,0.0,0.0,270.0,180.0,0.0,0.0,90.0,0.0
2,2026-03-16,RTBB1007,Beefsteak - 15lb 22ct Beef No1,Beefsteak,BB,MPL2_INV,MPL,0.0,733.0,270.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,270.0,0.0
3,2026-03-16,RTBB1007,Beefsteak - 15lb 22ct Beef No1,Beefsteak,BB,MPL3,MPL,0.0,249.0,249.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,249.0,0.0
4,2026-03-16,RTBB1007,Beefsteak - 15lb 22ct Beef No1,Beefsteak,BB,MPL6-B,MPL,0.0,1072.0,999.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,999.0,0.0


In [ ]:
# ── Cell 3 · Authenticate and render interactive Power BI report ──────────────
#
# DeviceCodeLoginAuthentication opens a browser tab where you enter the
# one-time code shown in the output below. Sign in with any account that has
# a Power BI Pro or Premium-per-user licence.
#
# QuickVisualize uploads the DataFrame as a push dataset to Power BI,
# auto-generates a report, and renders it inline in this notebook cell.

# Step 1 – authenticate
auth = DeviceCodeLoginAuthentication()

# Step 2 – build the dataset config from the flattened DataFrame
dataset_create_config = get_dataset_config(df_viz)

# Step 3 – create and display the interactive report
quick_visualize = QuickVisualize(dataset_create_config, auth=auth)
quick_visualize

Performing device flow authentication. Please follow the instructions below.
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code IQ3ZPLYD3 to authenticate.
